In [ ]:
"""
Task 1
"""

from pathlib import Path
import re


def read_log_file(file_path: Path) -> tuple[str, str]:
    """
    Read a log file and automatically detect its encoding.

    The function first attempts to read the file using UTF-8.
    If that fails, it falls back to ISO-8859-1 (Latin-1).

    Parameters
    ----------
    file_path : Path
        Path to the log file.

    Returns
    -------
    tuple[str, str]
        A tuple containing:
        - The file content as a string.
        - The encoding used to read the file.
    """

    try:
        # Try reading the file as UTF-8
        content = file_path.read_text(encoding="utf-8")
        return content, "utf-8"

    except UnicodeDecodeError:
        # Fall back to ISO-8859-1 if UTF-8 decoding fails
        content = file_path.read_text(encoding="iso-8859-1")
        return content, "iso-8859-1"


def parse_log_entries(file_text: str) -> list[dict]:
    """
    Extract valid transaction records from a log file.

    The function parses transaction lines and ignores any
    unrelated system messages such as printer warnings,
    debug messages, and network status logs.

    Parameters
    ----------
    file_text : str
        Raw text content from a log file.

    Returns
    -------
    list[dict]
        A list of transaction dictionaries containing:
        timestamp, register_id, item, price,
        currency and status.
    """

    # Match valid transaction records generated by the dataset generator
    pattern = re.compile(
        r"\[(.*?)\]\s+\[(.*?)\]\s+item=(.*?)\s+price=(\d+\.\d+)\s+(EUR|NOK|USD)\s+\((SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions = []

    # Process one log line at a time
    for line in file_text.splitlines():

        # Check whether the line matches the transaction format
        match = pattern.search(line)

        if match:

            # Create a transaction dictionary from the captured groups
            transaction = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item": match.group(3),
                "price": float(match.group(4)),
                "currency": match.group(5),
                "status": match.group(6),
            }

            transactions.append(transaction)

    return transactions

# Simple test code for verifying Task 1 functionality
if __name__ == "__main__":

    # Example test file
    file_path = Path(
    "d2_data/2026/library_cafe/POS_01/register_sales.log"
)

    # Read log file
    file_text, encoding = read_log_file(file_path)

    print(f"Encoding used: {encoding}")

    # Parse transactions
    transactions = parse_log_entries(file_text)

    print(f"Transactions found: {len(transactions)}")

    # Display the first three transactions
    for transaction in transactions[:3]:
        print(transaction)

Encoding used: utf-8
Transactions found: 80
{'timestamp': '2026-10-15 07:00:34', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 3.46, 'currency': 'EUR', 'status': 'REFUND'}
{'timestamp': '2026-10-15 07:02:51', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 3.46, 'currency': 'USD', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 07:03:45', 'register_id': 'POS_01', 'item': 'Cappuccino', 'price': 3.71, 'currency': 'EUR', 'status': 'SUCCESS'}


In [ ]:
"""
Task 2
"""
from pathlib import  Path

testpath = Path(r"C:\Users\jacob\OneDrive\Skrivebord\Vibecoding\INF_201\inf201_deliverable_2\data")
print(type(testpath))
def task_2(directory: str | Path) -> list[Path]:
    # 1. Convert input path to a pathlib.Path object
    path = Path(directory)

    log_files: list[Path] = []

    for item in path.iterdir():
        # 3. If item is a directory, recursively search inside it
        if item.is_dir():
            log_files.extend(task_2(item))

        # 4. If item is a .log file, collect it
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files

print(task_2(testpath))

<class 'pathlib.WindowsPath'>


FileNotFoundError: [WinError 3] Systemet finner ikke angitt bane: 'C:\\Users\\jacob\\OneDrive\\Skrivebord\\Vibecoding\\INF_201\\inf201_deliverable_2\\data'

In [ ]:
"""
Task 3
"""

import numpy as np


def remove_price_outliers(prices: np.ndarray) -> np.ndarray:
    """
    Replace price outliers with the median price.

    Outliers are identified using the three-sigma rule.
    If the three-sigma rule fails to detect obvious extreme
    values, values larger than 100 are also treated as outliers.

    Parameters
    ----------
    prices : np.ndarray
        One-dimensional array containing transaction prices.

    Returns
    -------
    np.ndarray
        Array with outliers replaced by the median value.
    """

    # Convert input to a NumPy float array
    prices = np.asarray(prices, dtype=float)

    # Calculate statistics
    mean = np.mean(prices)
    std = np.std(prices)

    # Identify outliers using the three-sigma rule
    outlier_mask = np.abs(prices - mean) > (3 * std)

    # Dataset generator uses 999.00 as an obvious bad value
    outlier_mask |= prices > 100

    # Calculate median of valid prices
    median_price = np.median(prices[~outlier_mask])

    # Create a copy of the original array
    cleaned_prices = prices.copy()

    # Replace outliers with the median value
    cleaned_prices[outlier_mask] = median_price

    return cleaned_prices


def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Compute refund ratios element-wise.

    The refund ratio is calculated as:

        refund_count / (sales_count + 1e-6)

    Parameters
    ----------
    sales_count : np.ndarray
        Number of sales.

    refund_count : np.ndarray
        Number of refunds.

    Returns
    -------
    np.ndarray
        Array containing refund ratios.
    """

    # Convert inputs to NumPy float arrays
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)

    # Avoid division by zero
    return refund_count / (sales_count + 1e-6)


if __name__ == "__main__":

    # Example test for Task 3
    prices = np.array([4.5, 5.0, 4.8, 999.0, 5.2])

    print("Original prices:")
    print(prices)

    print("\nCleaned prices:")
    print(remove_price_outliers(prices))

    sales = np.array([100, 50, 0])
    refunds = np.array([10, 5, 1])

    print("\nRefund ratios:")
    print(compute_refund_ratio(sales, refunds))

Original prices:
[  4.5   5.    4.8 999.    5.2]

Cleaned prices:
[4.5 5.  4.8 4.9 5.2]

Refund ratios:
[9.9999999e-02 9.9999998e-02 1.0000000e+06]


In [ ]:
"""
Task 4
"""

In [ ]:
"""
Task 5
"""

from pathlib import Path
import pandas as pd


def run_pipeline(data_directory: Path) -> None:
    """
    Execute the complete sales processing pipeline.

    The pipeline:
    - Finds all log files
    - Reads and parses transactions
    - Cleans price data
    - Applies currency correction
    - Generates analysis outputs
    - Saves results to disk

    Parameters
    ----------
    data_directory : Path
        Root directory containing the dataset.
    """

    # Find all log files
    log_files = task_2(data_directory)

    # Find all active register IDs from the configuration file
    active_registers = task_2_2("data/store_config.json")

    all_transactions = []

    # Read and parse every log file
    for log_file in log_files:

        file_text, _ = read_log_file(log_file)

        transactions = parse_log_entries(file_text)

        all_transactions.extend(transactions)

    # Create DataFrame
    df = task_4(all_transactions)

    # Keep only transactions from active registers
    df = df[df["register_id"].isin(active_registers)]

    # Clean prices using Task 3
    df["price"] = remove_price_outliers(df["price"].to_numpy())

    # Apply NOK currency correction
    df.loc[df["currency"] == "NOK", "price"] *= 10.0

    # Save cleaned transactions
    df.to_csv("cleaned_transactions.csv", index=False)

    # Generate hourly revenue report
    hourly_revenue = hour_sum(df)

    hourly_revenue.to_json(
        "hourly_revenue.json",
        orient="table"
    )

    # Generate status pivot table
    pivot_table_v = pivot_table(df)

    pivot_table_v.to_csv("status_pivot_table.csv")

    # Generate refund ratios
    refund_ratios = ref_per_item(df)

    refund_ratios.to_csv(
        "item_refund_ratios.csv",
        index=True
    )

    print("Pipeline completed successfully.")


if __name__ == "__main__":

    # Example execution
    data_directory = Path("data")

    run_pipeline(data_directory)

ModuleNotFoundError: No module named 'Deliverable_2_Task_1'